### Aula 06 — Agregações e execução distribuída

Esta aula transforma vendas em resumos por categoria, estado, forma de pagamento e conjunto completo. O objetivo é entender a **granularidade de cada resultado** e ler como o Spark combina cálculos locais com redistribuição e consolidação final.

As primeiras células importam `SparkSession`, disponibilizam as funções SQL como `F` e obtêm a sessão. Os exemplos usam o CSV `06_agregacoes/vendas.csv`, que deve existir no caminho preservado no código.

**Convenção de leitura:** as tabelas e planos abaixo são outputs históricos já presentes no arquivo. Os números são específicos desse conjunto de dados; não houve nova execução para produzir esta documentação. `groupBy(...).agg(...)` e `df.agg(...)` constroem DataFrames de resultado de forma lazy; `show()` dispara o cálculo necessário à exibição. `explain(True)` descreve o planejamento, sem solicitar todas as linhas da consulta.

In [21]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

In [22]:
spark = (
    SparkSession.builder
    .appName("aula_06_agregacoes")
    .getOrCreate()
)

#### 1. Ler as vendas e reconhecer sua granularidade

A leitura usa cabeçalho e inferência de tipos; a inferência pode realizar trabalho de leitura. `show()` exibe uma amostra de 20 linhas com identificadores, produto, categoria, quantidade, valores, localização e forma de pagamento. Cada linha é tratada pelo código como um registro de venda; agregar mudará essa granularidade.

Nos planos salvos, `valor_total` é `double`. Caudas como `4068842.090000002` refletem a representação de ponto flutuante e a acumulação numérica; a ordem de combinação distribuída pode alterar os últimos dígitos. Esses outputs foram preservados. Em uma modelagem financeira, tipo numérico e regra de arredondamento precisam ser definidos explicitamente; esta aula estuda o mecanismo de agregação.

In [23]:
df = spark.read.csv("/workspace/datasets/06_agregacoes/vendas.csv", header= True, inferSchema= True)
df.show()

+--------+----------+-----------+--------------------+--------------------+----------+--------------+-----------+--------------------+------+---------------+
|venda_id|data_venda|vendedor_id|             produto|           categoria|quantidade|valor_unitario|valor_total|              cidade|estado|forma_pagamento|
+--------+----------+-----------+--------------------+--------------------+----------+--------------+-----------+--------------------+------+---------------+
|       1|2025-01-01|        106|   Kit Halteres 20kg|     Esporte e Lazer|         3|         259.9|      779.7|              Recife|    PE| Cartão Crédito|
|       2|2025-01-01|        106|Carregador Turbo ...|Celulares e Acess...|         5|         80.91|     404.55|           São Paulo|    SP|         Boleto|
|       3|2025-01-01|        110|Console de Videogame|               Games|         1|        2999.0|     2999.0|            Salvador|    BA|  Cartão Débito|
|       4|2025-01-01|        110|    Notebook Core i

#### 2. Faturamento por categoria

`groupBy("categoria")` define a chave do resumo; `sum("valor_total")` soma os valores de cada grupo; `alias("faturamento_total")` nomeia a métrica. O resultado esperado é **uma linha por categoria encontrada**, e não uma linha por venda.

**Resultado salvo:** a tabela apresenta seis categorias com seus faturamentos. Não há `orderBy`, portanto a ordem mostrada não é uma classificação por faturamento.

Vendas de uma mesma categoria podem estar em partições diferentes. Reunir suas contribuições exige uma distribuição compatível com a chave: essa dependência torna a agregação agrupada tipicamente wide. Para Engenharia de Dados, explicitar a granularidade evita somar novamente valores já consolidados como se fossem registros individuais.

In [24]:
df_agregado = (
    df.groupBy("categoria")
    .agg(F.sum("valor_total").alias("faturamento_total"))
)

In [25]:
df_agregado.show()

+--------------------+------------------+
|           categoria| faturamento_total|
+--------------------+------------------+
|              Móveis| 4068842.090000002|
|     Esporte e Lazer|3568416.5300000054|
|               Games| 3341485.670000002|
|         Informática|  3570334.61999996|
|Celulares e Acess...| 2450861.539999995|
|    Eletrodomésticos| 5686772.190000008|
+--------------------+------------------+



##### 2.1. Ler `partial → Exchange → final`

O plano físico salvo mostra, de baixo para cima:

1. **`FileScan csv [categoria, valor_total]`**: o leitor fornece apenas as colunas necessárias à consulta. O `Project` do plano otimizado e o `ReadSchema` reduzido evidenciam Column Pruning; isso não converte o CSV em formato colunar nem elimina necessariamente seus bytes de leitura.
2. **`HashAggregate ... partial_sum(valor_total)`**: cada partição acumula uma soma intermediária por categoria. Assim, em vez de transferir todas as vendas, pode transferir menos estados parciais; o benefício depende de quantas chaves se repetem localmente.
3. **`Exchange hashpartitioning(categoria, 200)`**: distribui os estados pela chave para que contribuições da mesma categoria possam ser consolidadas. O shuffle movimenta esses dados; quem soma são os agregadores.
4. **`HashAggregate ... sum(valor_total)`**: combina os estados intermediários e produz o faturamento final por categoria.

O `200` é o número de partições de shuffle previsto nesse plano, não a quantidade de categorias, executors ou tasks simultâneas. `AdaptiveSparkPlan isFinalPlan=false` indica que a representação exibida ainda não é um plano adaptativo final; o número de partições efetivamente usadas pode ser ajustado pelo AQE. O output não inclui métricas de rede ou duração que permitam quantificar esse custo.

In [26]:
df_agregado.explain(True)

== Parsed Logical Plan ==
'Aggregate ['categoria], ['categoria, sum('valor_total) AS faturamento_total#493]
+- Relation [venda_id#402,data_venda#403,vendedor_id#404,produto#405,categoria#406,quantidade#407,valor_unitario#408,valor_total#409,cidade#410,estado#411,forma_pagamento#412] csv

== Analyzed Logical Plan ==
categoria: string, faturamento_total: double
Aggregate [categoria#406], [categoria#406, sum(valor_total#409) AS faturamento_total#493]
+- Relation [venda_id#402,data_venda#403,vendedor_id#404,produto#405,categoria#406,quantidade#407,valor_unitario#408,valor_total#409,cidade#410,estado#411,forma_pagamento#412] csv

== Optimized Logical Plan ==
Aggregate [categoria#406], [categoria#406, sum(valor_total#409) AS faturamento_total#493]
+- Project [categoria#406, valor_total#409]
   +- Relation [venda_id#402,data_venda#403,vendedor_id#404,produto#405,categoria#406,quantidade#407,valor_unitario#408,valor_total#409,cidade#410,estado#411,forma_pagamento#412] csv

== Physical Plan ==


#### 3. Cinco métricas no mesmo resumo por estado

`df_agregado2` produz uma linha por `estado`, com faturamento (`sum`), ticket médio (`avg`), quantidade de registros (`count("*")`), maior venda (`max`) e menor venda (`min`). A contagem é de linhas, não de unidades de produto: `quantidade` não é somada neste bloco.

`count("*")` inclui todas as linhas do grupo. Já `count(coluna)` conta somente valores não nulos, e `sum`, `avg`, `min` e `max` ignoram entradas nulas da expressão; grupos sem valor válido podem produzir `NULL` nessas métricas. O ticket médio é a média de `valor_total` por registro com valor não nulo, não uma média de preços unitários.

**Exemplo observado:** SP tem `1990` registros e BA tem `1866`. Esses valores descrevem a carga salva, sem impor uma distribuição uniforme. Calcular métricas compatíveis em um mesmo `agg` permite compartilhar a organização dos dados pela chave.

In [27]:
df_agregado2 = (
    df.groupBy("estado")
    .agg(
        F.sum("valor_total").alias("faturamento_total"),
        F.avg("valor_total").alias("ticket_medio"),
        F.count("*").alias("quantidade_vendas"),
        F.max("valor_total").alias("maior_venda"),
        F.min("valor_total").alias("menor_venda")
    )
)

In [28]:
df_agregado2.show()

+------+------------------+------------------+-----------------+-----------+-----------+
|estado| faturamento_total|      ticket_medio|quantidade_vendas|maior_venda|menor_venda|
+------+------------------+------------------+-----------------+-----------+-----------+
|    GO|1173497.0300000012| 2230.982946768063|              526|     6998.0|       79.9|
|    SP| 4492084.900000003|2257.3290954773884|             1990|     6998.0|      67.92|
|    RS|1238966.5800000012|2290.1415526802243|              541|     6998.0|       75.9|
|    MG|2772609.1599999964|2295.2062582781427|             1208|     6998.0|      67.92|
|    BA| 4296269.199999985|2302.3950696677307|             1866|     6998.0|      67.92|
|    PE|1389734.9500000014| 2285.748273026318|              608|     6998.0|       79.9|
|    CE|1485962.2399999998| 2227.829445277361|              667|     6998.0|      67.92|
|    RJ| 2831397.109999993|2240.0293591772097|             1264|     6998.0|      71.91|
|    DF|1475456.60999

##### 3.1. Um Exchange para métricas com a mesma chave

O plano salvo tem um `Exchange hashpartitioning(estado, 200)` entre dois `HashAggregate`, mesmo calculando cinco métricas. As funções parciais `partial_sum`, `partial_avg`, `partial_count`, `partial_max` e `partial_min` mantêm estados locais para cada estado e compartilham essa redistribuição.

Para a média, o estágio parcial conserva **soma e contagem de valores válidos**; a etapa final combina esses estados e divide o total pela contagem total. Tirar a média simples de médias de partições seria incorreto quando elas tivessem quantidades diferentes de registros. Para máximo e mínimo, basta combinar os extremos parciais.

O `FileScan` contém somente `valor_total` e `estado`, pois as demais colunas não participam dessas métricas. O plano demonstra compartilhamento do shuffle nesta consulta; não permite concluir que qualquer combinação de agregações sempre terá apenas um Exchange, como mostrará o caso distinto.

In [29]:
df_agregado2.explain(True)

== Parsed Logical Plan ==
'Aggregate ['estado], ['estado, sum('valor_total) AS faturamento_total#523, avg('valor_total) AS ticket_medio#525, count(1) AS quantidade_vendas#527L, max('valor_total) AS maior_venda#529, min('valor_total) AS menor_venda#531]
+- Relation [venda_id#402,data_venda#403,vendedor_id#404,produto#405,categoria#406,quantidade#407,valor_unitario#408,valor_total#409,cidade#410,estado#411,forma_pagamento#412] csv

== Analyzed Logical Plan ==
estado: string, faturamento_total: double, ticket_medio: double, quantidade_vendas: bigint, maior_venda: double, menor_venda: double
Aggregate [estado#411], [estado#411, sum(valor_total#409) AS faturamento_total#523, avg(valor_total#409) AS ticket_medio#525, count(1) AS quantidade_vendas#527L, max(valor_total#409) AS maior_venda#529, min(valor_total#409) AS menor_venda#531]
+- Relation [venda_id#402,data_venda#403,vendedor_id#404,produto#405,categoria#406,quantidade#407,valor_unitario#408,valor_total#409,cidade#410,estado#411,forma_

#### 4. Agrupamento com chave composta

`groupBy("estado", "forma_pagamento")` aumenta o detalhe do resumo: cada linha corresponde a uma **combinação** de estado e meio de pagamento. As métricas continuam sendo quantidade de registros e soma de `valor_total`.

A amostra salva mostra, por exemplo, `(BA, Pix)` com `575` vendas e faturamento próximo de `1.292.723,23`. O `show()` limita a saída a 20 linhas; não permite inferir quantas combinações existem no total. Essa escolha de granularidade é útil para preparar uma tabela agregada destinada a análises por região e canal de pagamento.

In [30]:
df_agregado3 = (
    df.groupBy("estado", "forma_pagamento")
    .agg(
        F.count("*").alias("quantidade_vendas"),
        F.sum("valor_total").alias("faturamento_total")
    )
        
    )


In [31]:
df_agregado3.show()

+------+---------------+-----------------+------------------+
|estado|forma_pagamento|quantidade_vendas| faturamento_total|
+------+---------------+-----------------+------------------+
|    PR|            Pix|              190|         398349.74|
|    CE|  Cartão Débito|              138|         291080.76|
|    CE|         Boleto|              145|344054.85999999987|
|    DF|            Pix|              202| 475971.4400000001|
|    PE|  Cartão Débito|              129|301570.50000000006|
|    SP|         Boleto|              400| 915608.9700000001|
|    SP|  Cartão Débito|              419| 911452.7400000006|
|    RS| Cartão Crédito|              165| 365503.5999999998|
|    MG|            Pix|              322|         720718.65|
|    BA|            Pix|              575|1292723.2300000014|
|    RJ|  Cartão Débito|              266|         576855.03|
|    GO|         Boleto|              107|213069.55999999994|
|    SP| Cartão Crédito|              593|1349837.4699999997|
|    PE|

##### 4.1. A chave composta no plano físico

Neste output, os dois agregadores usam `keys=[estado, forma_pagamento]`, e o Exchange usa `hashpartitioning(estado, forma_pagamento, 200)`. Assim, a mesma combinação chega a uma distribuição apropriada para consolidar seus estados parciais. Estados iguais com meios de pagamento diferentes continuam sendo grupos diferentes.

`partial_count(1)` e `partial_sum(valor_total)` fazem a redução local; `count(1)` e `sum(valor_total)` combinam esses estados ao final. `count(1)` é a representação planejada da contagem de todas as linhas. O `ReadSchema` se restringe a `valor_total`, `estado` e `forma_pagamento`.

In [32]:
df_agregado3.explain(True)

== Parsed Logical Plan ==
'Aggregate ['estado, 'forma_pagamento], ['estado, 'forma_pagamento, count(1) AS quantidade_vendas#599L, sum('valor_total) AS faturamento_total#601]
+- Relation [venda_id#402,data_venda#403,vendedor_id#404,produto#405,categoria#406,quantidade#407,valor_unitario#408,valor_total#409,cidade#410,estado#411,forma_pagamento#412] csv

== Analyzed Logical Plan ==
estado: string, forma_pagamento: string, quantidade_vendas: bigint, faturamento_total: double
Aggregate [estado#411, forma_pagamento#412], [estado#411, forma_pagamento#412, count(1) AS quantidade_vendas#599L, sum(valor_total#409) AS faturamento_total#601]
+- Relation [venda_id#402,data_venda#403,vendedor_id#404,produto#405,categoria#406,quantidade#407,valor_unitario#408,valor_total#409,cidade#410,estado#411,forma_pagamento#412] csv

== Optimized Logical Plan ==
Aggregate [estado#411, forma_pagamento#412], [estado#411, forma_pagamento#412, count(1) AS quantidade_vendas#599L, sum(valor_total#409) AS faturamento_

#### 5. Agregação global: um resumo para toda a base

`df.agg(...)`, sem chaves de agrupamento, produz uma única linha com faturamento, valor médio e quantidade total de registros. Trata-se de outra granularidade: não há estado ou categoria identificando grupos separados.

**Resultado salvo:** `quantidade_total = 10000`, faturamento `2.2686712640000276E7` e valor médio `2268.671264000028`. A notação `E7` multiplica o número por dez elevado à sétima potência; o faturamento corresponde aproximadamente a `22.686.712,64`. São valores observados no output existente, sem recálculo.

In [33]:
df_global = (
    df.agg(
        F.sum("valor_total").alias("faturamento_total"),
        F.avg("valor_total").alias("valor_medio"),
        F.count("*").alias("quantidade_total")
    )
)

In [34]:
df_global.show()

+--------------------+-----------------+----------------+
|   faturamento_total|      valor_medio|quantidade_total|
+--------------------+-----------------+----------------+
|2.2686712640000276E7|2268.671264000028|           10000|
+--------------------+-----------------+----------------+



##### 5.1. O significado de `Exchange SinglePartition`

No plano salvo, ambos os `HashAggregate` exibem `keys=[]`. O parcial calcula soma, soma/contagem para a média e contagem de linhas em cada partição de entrada. O **`Exchange SinglePartition`** reúne os estados parciais em uma partição para a consolidação do único resultado final.

Isso não obriga a leitura e todo o cálculo anterior a acontecerem em uma só task, nem significa enviar necessariamente todas as vendas brutas para o final: há redução parcial antes do Exchange. A etapa final usa uma partição, mas o volume que ela recebe depende dos estados intermediários. Em Engenharia de Dados, separar paralelismo de leitura de paralelismo de consolidação ajuda a localizar onde uma consulta pode concentrar trabalho.

In [35]:
df_global.explain(True)

== Parsed Logical Plan ==
'Aggregate [sum('valor_total) AS faturamento_total#633, avg('valor_total) AS valor_medio#635, count(1) AS quantidade_total#637L]
+- Relation [venda_id#402,data_venda#403,vendedor_id#404,produto#405,categoria#406,quantidade#407,valor_unitario#408,valor_total#409,cidade#410,estado#411,forma_pagamento#412] csv

== Analyzed Logical Plan ==
faturamento_total: double, valor_medio: double, quantidade_total: bigint
Aggregate [sum(valor_total#409) AS faturamento_total#633, avg(valor_total#409) AS valor_medio#635, count(1) AS quantidade_total#637L]
+- Relation [venda_id#402,data_venda#403,vendedor_id#404,produto#405,categoria#406,quantidade#407,valor_unitario#408,valor_total#409,cidade#410,estado#411,forma_pagamento#412] csv

== Optimized Logical Plan ==
Aggregate [sum(valor_total#409) AS faturamento_total#633, avg(valor_total#409) AS valor_medio#635, count(1) AS quantidade_total#637L]
+- Project [valor_total#409]
   +- Relation [venda_id#402,data_venda#403,vendedor_id#

#### 6. Métricas condicionais sem eliminar as demais vendas

`df_condicional` mantém todas as linhas de cada estado e calcula, no mesmo resumo, quantidade/faturamento totais, quantidade/faturamento de Pix e faturamento das linhas cuja cidade é Salvador. Diferentemente de um `filter` antes do agrupamento, as condições de `when` afetam apenas suas respectivas métricas.

Para contar Pix, `when(..., 1).otherwise(0)` atribui uma contribuição de um às linhas correspondentes e zero às demais; `sum` acumula essas contribuições. Para o faturamento, a contribuição é `valor_total` ou zero. Usar `count` sobre uma expressão com `otherwise(0)` contaria também os zeros, pois eles não são nulos.

**Resultado salvo:** BA registra `575` vendas Pix e faturamento de Salvador próximo de `1.955.731,98`; os demais estados exibidos têm zero nessa última métrica. Isso descreve os dados e a condição literal do código. A expressão não faz padronização de nomes de cidades nem valida se sua associação ao estado está correta.

In [36]:
df_condicional = (
    df.groupBy("estado")
    .agg(
        F.count("*").alias("quantidade_total"),
        F.sum("valor_total").alias("faturamento_total"),
        F.sum(F.when(F.col("forma_pagamento") == "Pix", 1).otherwise(0)).alias("quantidade_pix"),
        F.sum(F.when(F.col("forma_pagamento") == "Pix", F.col("valor_total")).otherwise(0)).alias("faturamento_pix"),
        F.sum(F.when(F.col("cidade") == "Salvador", F.col("valor_total")).otherwise(0)).alias("faturamento_total_salvador")
        
    )
)

In [37]:
df_condicional.show()

+------+----------------+------------------+--------------+------------------+--------------------------+
|estado|quantidade_total| faturamento_total|quantidade_pix|   faturamento_pix|faturamento_total_salvador|
+------+----------------+------------------+--------------+------------------+--------------------------+
|    GO|             526|1173497.0300000012|           172|412100.91999999987|                       0.0|
|    SP|            1990| 4492084.900000003|           578|1315185.7199999986|                       0.0|
|    RS|             541|1238966.5800000012|           164| 381641.0099999998|                       0.0|
|    MG|            1208|2772609.1599999964|           322|         720718.65|                       0.0|
|    BA|            1866| 4296269.199999985|           575|1292723.2300000014|        1955731.9800000004|
|    PE|             608|1389734.9500000014|           190|434953.82999999996|                       0.0|
|    CE|             667|1485962.2399999998|  

##### 6.1. Condições dentro das agregações

O plano traduz `when/otherwise` para `CASE WHEN ... THEN ... ELSE ... END`, dentro das funções `partial_sum` e `sum`. As condições são avaliadas para formar contribuições locais, que seguem pelo mesmo `Exchange hashpartitioning(estado, 200)` das demais métricas.

O `FileScan` agora precisa de `valor_total`, `cidade`, `estado` e `forma_pagamento`. `DataFilters` e `PushedFilters` permanecem vazios: a consulta não pode descartar todas as linhas não Pix, porque elas ainda participam dos totais. Esse contraste ajuda a distinguir filtragem da base de uma regra aplicada somente a uma métrica.

In [38]:
df_condicional.explain(True)

== Parsed Logical Plan ==
'Aggregate ['estado], ['estado, count(1) AS quantidade_total#692L, sum('valor_total) AS faturamento_total#694, sum(CASE WHEN ('forma_pagamento = Pix) THEN 1 ELSE 0 END) AS quantidade_pix#696, sum(CASE WHEN ('forma_pagamento = Pix) THEN 'valor_total ELSE 0 END) AS faturamento_pix#698, sum(CASE WHEN ('cidade = Salvador) THEN 'valor_total ELSE 0 END) AS faturamento_total_salvador#700]
+- Relation [venda_id#402,data_venda#403,vendedor_id#404,produto#405,categoria#406,quantidade#407,valor_unitario#408,valor_total#409,cidade#410,estado#411,forma_pagamento#412] csv

== Analyzed Logical Plan ==
estado: string, quantidade_total: bigint, faturamento_total: double, quantidade_pix: bigint, faturamento_pix: double, faturamento_total_salvador: double
Aggregate [estado#411], [estado#411, count(1) AS quantidade_total#692L, sum(valor_total#409) AS faturamento_total#694, sum(CASE WHEN (forma_pagamento#412 = Pix) THEN 1 ELSE 0 END) AS quantidade_pix#696L, sum(CASE WHEN (forma_pa

#### 7. Contagem exata de vendedores distintos por estado

`countDistinct("vendedor_id")` conta IDs não nulos diferentes dentro de cada estado. O vendedor que aparece em várias vendas do mesmo estado contribui uma única vez; se atua em estados diferentes, pode ser contado em cada um deles.

**Resultado salvo:** cada um dos dez estados exibidos tem `20` vendedores distintos. Somar esses resultados não produz necessariamente o número global de vendedores: o mesmo ID pode aparecer em mais de um estado.

Também seria incorreto calcular contagens distintas independentes em cada partição e simplesmente somá-las. Um vendedor presente em duas partições do mesmo estado seria contado duas vezes. O próximo plano mostra como o Spark trata essa sobreposição, sem usar uma aproximação estatística.

In [41]:
df_distinct = (
    df.groupBy("estado")
    .agg(
        F.countDistinct("vendedor_id").alias("quantidade_vendedores_distintos")
    )
)

In [42]:
df_distinct.show()

+------+-------------------------------+
|estado|quantidade_vendedores_distintos|
+------+-------------------------------+
|    GO|                             20|
|    SP|                             20|
|    RS|                             20|
|    MG|                             20|
|    BA|                             20|
|    PE|                             20|
|    CE|                             20|
|    RJ|                             20|
|    DF|                             20|
|    PR|                             20|
+------+-------------------------------+



##### 7.1. Por que o plano distinto tem dois Exchanges?

Leia os operadores salvos de baixo para cima:

1. `FileScan` lê `vendedor_id` e `estado`.
2. `HashAggregate(keys=[estado, vendedor_id], functions=[])` elimina repetições locais do par, mesmo sem calcular uma soma ou contagem nessa etapa.
3. O primeiro `Exchange hashpartitioning(estado, vendedor_id, 200)` aproxima pares iguais; outro `HashAggregate` com essas duas chaves consolida sua deduplicação entre partições.
4. `HashAggregate(keys=[estado], functions=[partial_count(distinct vendedor_id)])` prepara estados de contagem por estado sobre os pares deduplicados.
5. O segundo `Exchange hashpartitioning(estado, 200)` reúne essas contribuições; o último `HashAggregate` produz `count(distinct vendedor_id)` por estado.

O primeiro alinhamento é pelo **par**; o segundo é somente por **estado**. Esses requisitos explicam os dois Exchanges e as etapas adicionais em relação à soma por estado. Cardinalidade elevada pode tornar essa deduplicação e a manutenção de estados mais custosas; este notebook não contém medições para comparar os tempos das consultas.

In [43]:
df_distinct.explain(True)

== Parsed Logical Plan ==
'Aggregate ['estado], ['estado, 'count(distinct 'vendedor_id) AS quantidade_vendedores_distintos#793]
+- Relation [venda_id#402,data_venda#403,vendedor_id#404,produto#405,categoria#406,quantidade#407,valor_unitario#408,valor_total#409,cidade#410,estado#411,forma_pagamento#412] csv

== Analyzed Logical Plan ==
estado: string, quantidade_vendedores_distintos: bigint
Aggregate [estado#411], [estado#411, count(distinct vendedor_id#404) AS quantidade_vendedores_distintos#793L]
+- Relation [venda_id#402,data_venda#403,vendedor_id#404,produto#405,categoria#406,quantidade#407,valor_unitario#408,valor_total#409,cidade#410,estado#411,forma_pagamento#412] csv

== Optimized Logical Plan ==
Aggregate [estado#411], [estado#411, count(distinct vendedor_id#404) AS quantidade_vendedores_distintos#793L]
+- Project [vendedor_id#404, estado#411]
   +- Relation [venda_id#402,data_venda#403,vendedor_id#404,produto#405,categoria#406,quantidade#407,valor_unitario#408,valor_total#409,

#### 8. Encerramento

`spark.stop()` encerra a sessão/contexto e seus recursos. Para retomar o estudo em execução, é necessário obter uma sessão ativa antes de usar os DataFrames; as saídas salvas continuam sendo o registro histórico dos exemplos.

In [49]:
spark.stop()

#### Resumo dos aprendizados

| Consulta | Granularidade final | Redistribuição visível no plano salvo |
|---|---|---|
| Soma por categoria | Uma linha por categoria | `hashpartitioning(categoria, 200)` |
| Cinco métricas por estado | Uma linha por estado | Um Exchange compartilhado por estado |
| Estado e forma de pagamento | Uma linha por combinação | Hash das duas chaves |
| Resumo global | Uma linha para toda a base | `SinglePartition` após cálculo parcial |
| Métricas condicionais | Uma linha por estado | Estado, com `CASE WHEN` nos agregadores |
| Vendedores distintos | Uma linha por estado | Primeiro o par estado/vendedor, depois estado |

O padrão central é **agregar localmente → redistribuir estados → consolidar**. O parcial pode reduzir o volume do shuffle, o Exchange define a distribuição e o agregador final combina os estados. Média requer soma e contagem; contagem distinta requer tratar duplicidades que cruzam partições. Os planos com `isFinalPlan=false` representam estratégias exibidas naquele momento, não um relatório final de desempenho.

Veja [o capítulo de agregações](../docs/06_agregacoes.md) e [o Desafio 06](../desafios/desafio_06_agregacoes.ipynb) para continuar o estudo. As métricas e planos citados aqui pertencem às células desta aula.